# Phase B — Backbone Investigation (E02–E05) on MVTec LOCO AD (Combined)

Compares **4 modern backbones** integrated into the PUAD framework against verified **E01 Original PUAD** (`87.56%` Combined Mean / `83.30%` Pooled Raw):
- **E02**: `ResNet-18` (residual inductive bias)
- **E03**: `DenseNet-121` (dense feature reuse)
- **E04**: `EfficientNet-B0` (compound-scaled MBConv efficiency)
- **E05**: `ConvNeXt-Tiny` (modern large-kernel pure-CNN)

All 4 experiments use the **exact same controlled protocol**:
- Pooled 5-category `train/good` (`1,778`), `validation/good` (`305`), logical-only test (`1,136`, **0** structural).
- Frozen ImageNet-pretrained backbone teacher (`{C3, C4, C5}` → `384×32×32`) + randomly initialized trainable student + 1×1 global AutoEncoder + Ledoit-Wolf Mahalanobis + validation z-score fusion.
- `batch_size=8`, `AdamW`, `lr=3e-4` (cosine to `1e-5`), `200 epochs`, `patience=10`, hardware-native AMP (`BF16` on A100/L4, `FP16+GradScaler` on T4), auto-resumable & synced to Google Drive every epoch.

In [1]:
#@title 1. Section 3 — Hardware, Runtime, RAM & Disk Audit
import os, platform, shutil, sys, torch

print('=== COLAB SESSION AUDIT (PHASE B) ===')
print('Python        :', sys.version.split()[0])
print('PyTorch       :', torch.__version__, '| CUDA:', torch.version.cuda)
print('CUDA Available:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'No GPU assigned! Switch Runtime -> Change runtime type -> GPU.'
props = torch.cuda.get_device_properties(0)
native_bf16 = props.major >= 8 and torch.cuda.is_bf16_supported()
print(f'GPU Model     : {props.name} (CC {props.major}.{props.minor})')
print(f'GPU VRAM      : {props.total_memory / (1024**3):.2f} GiB | Native BF16: {native_bf16}')
import psutil
vm = psutil.virtual_memory()
print(f'System RAM    : {vm.available/(1024**3):.1f} GB free / {vm.total/(1024**3):.1f} GB total')
disk = shutil.disk_usage('/content')
print(f'Local SSD     : {disk.free/(1024**3):.1f} GB free / {disk.total/(1024**3):.1f} GB total')

=== COLAB SESSION AUDIT (PHASE B) ===
Python        : 3.13.16
PyTorch       : 2.14.1+cu130 | CUDA: 13.0
CUDA Available: True
GPU Model     : NVIDIA L4 (CC 8.9)
GPU VRAM      : 22.03 GiB | Native BF16: True
System RAM    : 50.8 GB free / 53.0 GB total
Local SSD     : 223.3 GB free / 235.7 GB total


In [2]:
#@title 2. Mount Drive, Pull Latest Code & Ensure Local SSD Datasets
import os, subprocess
from google.colab import drive
drive.mount('/content/drive')

DRIVE_PROJECT_DIR = '/content/drive/MyDrive/DL_Endsem_Team11'
DRIVE_LOCO_TAR    = f'{DRIVE_PROJECT_DIR}/data/mvtec_loco_anomaly_detection.tar.xz'
DRIVE_IMNET_ZIP   = f'{DRIVE_PROJECT_DIR}/data/imagenet_penalty_10k.zip'
DRIVE_TEACHER_PT  = f'{DRIVE_PROJECT_DIR}/weights/teacher_pdn_s.pt'
DRIVE_RESULTS_DIR = f'{DRIVE_PROJECT_DIR}/runs'
GITHUB_REPO_URL   = 'https://github.com/Chimera418/dl-endsem-logical-ad.git'

REPO_DIR = '/content/Endsem'
if os.path.exists(REPO_DIR):
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
else:
    subprocess.run(['git', 'clone', GITHUB_REPO_URL, REPO_DIR], check=True)

!pip install -q timm thop einops pyyaml psutil tqdm
%cd /content/Endsem

LOCAL_LOCO = '/content/data/mvtec_loco_anomaly_detection'
LOCAL_IMNET = '/content/data/imagenet_penalty'
os.makedirs('/content/data', exist_ok=True)
if not os.path.isdir(os.path.join(LOCAL_LOCO, 'breakfast_box')):
    os.makedirs(LOCAL_LOCO, exist_ok=True)
    !tar -xf "$DRIVE_LOCO_TAR" -C "$LOCAL_LOCO"
if not os.path.isdir(LOCAL_IMNET):
    !unzip -q "$DRIVE_IMNET_ZIP" -d /content/data/

os.environ['ENDSEM_LOCO_ROOT'] = LOCAL_LOCO
os.environ['ENDSEM_IMAGENET_ROOT'] = LOCAL_IMNET
os.environ['ENDSEM_TEACHER_PATH'] = DRIVE_TEACHER_PT
!python scripts/verify_env.py

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/Endsem
SECTION 3 — ENVIRONMENT & DATASET VERIFICATION AUDIT
1. CUDA Available    : True
2. GPU Model         : NVIDIA L4 (CC 8.9, BF16=True)
3. GPU Memory        : 22.03 GiB (22563 MiB)
4. PyTorch / CUDA    : PyTorch 2.14.1+cu130 | CUDA 13.0 | cuDNN 92800
   Python / Packages : Python 3.13.16 | timm=1.0.31 | thop=0.1.1 | mamba_ssm=None
5. System RAM        : 50.32 GB free / 52.96 GB total (5.0% used)
6. Disk Space        : 223.32 GB free / 235.68 GB total (/content/Endsem)
------------------------------------------------------------------------------
7. Dataset & Weights Accessibility:
   MVTec LOCO AD     : OK (/content/data/mvtec_loco_anomaly_detection)
     train=1778 | val=305 | test_good=575 | test_logical=561 | test_structural=432 (EXCLUDED)
   ImageNet Penalty  : OK (10,000 images at /content/data/imagenet_penalty)
   PDN-S Teacher     : OK (/

In [3]:
!git pull
!python scripts/run_experiment.py --config configs/E01_puad.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E02_resnet18.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E03_densenet121.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E04_efficientnet_b0.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E05_convnext_tiny.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RESULTS_DIR" --out-dir "$DRIVE_RESULTS_DIR/phaseB_summary"


Already up to date.
RUN: E01_PUAD_200ep | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  Penalty corpus ready: 10,000 images (batch_size=8)
  Computing streaming teacher channel normalization stats on pooled train/good...
  Starting training: epochs=200, batch_size=8, amp=torch.bfloat16, trainable_params=5,363,712, frozen_params=2,694,144
  Epoch   1/200 | train=10.2726 | val=7.3053 | lr=3.0e-04 | 48.9s <-- BEST
  Epoch   2/200 | train=6.2670 | val=5.3701 | lr=3.0e-04 | 39.5s <-- BEST
  Epoch   3/200 | train=4.9684 | val=4.4521 | lr=3.0e-04 | 39.5s <-- BEST
  Epoch   4/200 | train=4.2094 | val=3.9509 | lr=3.0e-04 | 39.4s <-- BEST
  Epoch   5/200 | train=3.8553 | val=3.6095 | lr=3.0e-04 | 39.6s <-- BEST
  Epoch   6/200 | train=3.5104 | val=3.3138 | lr=3.0e-04 | 39.4s <-- BEST
  Epoch   7/200 | train=3.2775 | val=3.22

In [10]:
!git pull
!python scripts/run_experiment.py --config configs/E02_resnet18.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E03_densenet121.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E04_efficientnet_b0.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/run_experiment.py --config configs/E05_convnext_tiny.yaml --mode train --force --drive-sync-dir "$DRIVE_RESULTS_DIR"
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RESULTS_DIR" --out-dir "$DRIVE_RESULTS_DIR/phaseB_summary"

remote: Enumerating objects: 9, done.
remote: Counting objects: 100% (9/9), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 5 (delta 4), reused 5 (delta 4), pack-reused 0 (from 0)
Unpacking objects: 100% (5/5), 721 bytes | 721.00 KiB/s, done.
From https://github.com/Chimera418/dl-endsem-logical-ad
   c75ab41..12f59f3  main       -> origin/main
Updating c75ab41..12f59f3
Fast-forward
 src/models/trainer.py | 27 +++++++++++----------------
 1 file changed, 11 insertions(+), 16 deletions(-)
RUN: E02_ResNet18 | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
Loading pooled train/good and validation/good (img_size=256, cache_in_ram=True)...
  Penalty corpus ready: 10,000 images (batch_size=8)
  Computing streaming teacher channel normalization stats on pooled train/good...
  Starting training: epochs=200, batch_size=8, amp=torch.bfloat16, trainable_params=12,913,344, frozen_params=11,176

In [11]:
#@title 3. Run E01 (200-Epoch Unified Control) — Original PUAD
!python scripts/run_experiment.py --config configs/E01_puad.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

RUN: E01_PUAD_200ep | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
[skip] Training already complete: /content/Endsem/experiments/E01_PUAD_200ep/training_summary.json
Loaded checkpoint: epoch=183 | best_val_loss=1.0517
  [Stage 2] Extracting pooled train/good features for Mahalanobis fit...
  [Stage 2] Calibrating z-score statistics on pooled validation/good...
  [Stage 2] Running single-pass evaluation on logical-only test set (1,136 images)...

=== E01_PUAD_200ep (train) ===
Per-Category Logical I-AUROC:
  Breakfast Box         :  90.53%
  Juice Bottle          :  96.87%
  Pushpins              :  98.02%
  Screw Bag             :  68.94%
  Splicing Connectors   :  93.85%
  Combined Mean AUROC   :  89.64%  (Primary)
  Pooled Raw AUROC      :  84.07%  (Secondary)
  Delta vs A2 PUAD      : Mean +2.08 pp | Pooled +0.77 pp
Efficiency: Params=8.058M (Trainable=5.364M) | GFLOPs=75.9066 | Latency=10.004 

In [12]:
#@title 4. Run E02 — ResNet-18
!python scripts/run_experiment.py --config configs/E02_resnet18.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

RUN: E02_ResNet18 | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
  [drive-restore] Restored train_resume.pth from /content/drive/MyDrive/DL_Endsem_Team11/runs/E02_ResNet18
[skip] Training already complete: /content/Endsem/experiments/E02_ResNet18/training_summary.json
Loaded checkpoint: epoch=9 | best_val_loss=30.7954
  [Stage 2] Extracting pooled train/good features for Mahalanobis fit...
  [Stage 2] Calibrating z-score statistics on pooled validation/good...
  [Stage 2] Running single-pass evaluation on logical-only test set (1,136 images)...

=== E02_ResNet18 (train) ===
Per-Category Logical I-AUROC:
  Breakfast Box         :  85.40%
  Juice Bottle          :  87.62%
  Pushpins              :  62.88%
  Screw Bag             :  64.51%
  Splicing Connectors   :  70.08%
  Combined Mean AUROC   :  74.10%  (Primary)
  Pooled Raw AUROC      :  69.95%  (Secondary)
  Delta vs A2 PUAD      : Mean -13.46

In [13]:
#@title 5. Run E03 — DenseNet-121
!python scripts/run_experiment.py --config configs/E03_densenet121.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

RUN: E03_DenseNet121 | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
  [drive-restore] Restored train_resume.pth from /content/drive/MyDrive/DL_Endsem_Team11/runs/E03_DenseNet121
[skip] Training already complete: /content/Endsem/experiments/E03_DenseNet121/training_summary.json
Loaded checkpoint: epoch=37 | best_val_loss=14.3004
  [Stage 2] Extracting pooled train/good features for Mahalanobis fit...
  [Stage 2] Calibrating z-score statistics on pooled validation/good...
  [Stage 2] Running single-pass evaluation on logical-only test set (1,136 images)...

=== E03_DenseNet121 (train) ===
Per-Category Logical I-AUROC:
  Breakfast Box         :  95.23%
  Juice Bottle          :  94.74%
  Pushpins              :  65.11%
  Screw Bag             :  59.85%
  Splicing Connectors   :  83.30%
  Combined Mean AUROC   :  79.65%  (Primary)
  Pooled Raw AUROC      :  78.03%  (Secondary)
  Delta vs A2 PUAD      

In [14]:
#@title 6. Run E04 — EfficientNet-B0
!python scripts/run_experiment.py --config configs/E04_efficientnet_b0.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

RUN: E04_EfficientNetB0 | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
Unexpected keys (bn2.num_batches_tracked, bn2.bias, bn2.running_mean, bn2.running_var, bn2.weight, classifier.bias, classifier.weight, conv_head.weight) found while loading pretrained weights. This may be expected if model is being adapted.
  [drive-restore] Restored train_resume.pth from /content/drive/MyDrive/DL_Endsem_Team11/runs/E04_EfficientNetB0
[skip] Training already complete: /content/Endsem/experiments/E04_EfficientNetB0/training_summary.json
Loaded checkpoint: epoch=132 | best_val_loss=5.8452
  [Stage 2] Extracting pooled train/good features for Mahalanobis fit...
  [Stage 2] Calibrating z-score statistics on pooled validation/good...
  [Stage 2] Running single-pass evaluation on logical-only test set (1,136 images)...

=== E04_EfficientNetB0 (train) ===
Per-Category Logical I-AUROC:
  Breakfast Box         :  90.90%

In [15]:
#@title 7. Run E05 — ConvNeXt-Tiny
!python scripts/run_experiment.py --config configs/E05_convnext_tiny.yaml --mode train --drive-sync-dir "$DRIVE_RESULTS_DIR"

RUN: E05_ConvNeXtTiny | mode=train | device=cuda | seed=42
Dataset audit OK: train=1778 | val=305 | test_good=575 | test_logical=561 | structural_excluded=432
  [drive-restore] Restored train_resume.pth from /content/drive/MyDrive/DL_Endsem_Team11/runs/E05_ConvNeXtTiny
[skip] Training already complete: /content/Endsem/experiments/E05_ConvNeXtTiny/training_summary.json
Loaded checkpoint: epoch=1 | best_val_loss=25.4076
  [Stage 2] Extracting pooled train/good features for Mahalanobis fit...
  [Stage 2] Calibrating z-score statistics on pooled validation/good...
  [Stage 2] Running single-pass evaluation on logical-only test set (1,136 images)...

=== E05_ConvNeXtTiny (train) ===
Per-Category Logical I-AUROC:
  Breakfast Box         :  87.48%
  Juice Bottle          :  90.65%
  Pushpins              :  56.94%
  Screw Bag             :  51.04%
  Splicing Connectors   :  73.44%
  Combined Mean AUROC   :  71.91%  (Primary)
  Pooled Raw AUROC      :  70.51%  (Secondary)
  Delta vs A2 PUAD   

In [16]:
#@title 8. Aggregate & Compare E01–E05 (Phase B Summary Table)
!python scripts/aggregate_results.py --dirs experiments "$DRIVE_RESULTS_DIR" --out-dir "$DRIVE_RESULTS_DIR/phaseB_summary"

| Run | BB | JB | PP | SB | SC | **Comb Mean** | **Pooled** | Δ Mean | Δ Pooled | Params (M) | GFLOPs | Lat (ms) | Peak VRAM (MiB) | GPU |
| :--- | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :---: | :--- |
| E01_PUAD_200ep | 90.53% | 96.87% | 98.02% | 68.94% | 93.85% | **89.64%** | **84.07%** | +2.08 pp | +0.77 pp | 8.058 | 75.9066 | 10.004 | 1463.44 | NVIDIA L4 |
| E01_PUAD_verify_a2 | 90.99% | 97.58% | 86.73% | 67.87% | 94.64% | **87.56%** | **83.30%** | +0.00 pp | +0.00 pp | 8.058 | 75.9066 | 20.48 | 171.78 | NVIDIA GeForce RTX 4060 Laptop GPU |
| E02_ResNet18 | 85.40% | 87.62% | 62.88% | 64.51% | 70.08% | **74.10%** | **69.95%** | -13.46 pp | -13.36 pp | 24.09 | 14.5346 | 7.507 | 912.24 | NVIDIA L4 |
| E03_DenseNet121 | 95.23% | 94.74% | 65.11% | 59.85% | 83.30% | **79.65%** | **78.03%** | -7.91 pp | -5.27 pp | 16.284 | 21.119 | 33.69 | 2635.03 | NVIDIA L4 |
| E04_EfficientNetB0 | 90.90% | 96.88% | 66.41% | 65.27% | 85.58% | **81